# Widgets
Są to obiekty w Pythonie reagujące na zdarzenia i umożliwiające obsługę różnych popularnych kontrolek w przeglądarce. Wykorzystane zostaną do utworzenia prostego GUI do obsługi robota. Do ich obsługi oknieczne jest zaimportowanie biblioteki ipywidgets.

In [ ]:
import ipywidgets as widgets  # zamiast korzystania z pełnej 
# nazwy biblioteki ipywidgets wykorzystany zostanie alias o 
# nazwie widgets.


In [ ]:
import rospy
from geometry_msgs.msg import Twist


In [ ]:
rospy.init_node("ipywidgets_controller")


Na zajęciach w których omawiany był Publisher wysyłanie prędkości sterujących odbywało się w następuący sposób:

In [ ]:
# obiekt Publishera
twist_publisher= rospy.Publisher("/turtle1/cmd_vel",Twist,
                                 queue_size=10)


In [ ]:
# wysłanie wiadomości
some_message=Twist()
some_message.angular.z=1
some_message.linear.x=1

twist_publisher.publish(some_message)


In [ ]:
# Można też utworzyć funkcję, która będzie przyjmowała argumenty
# związane z prędkością postępową i obrotową robota i wysyłała do 
# ROS odpowiednią komendę na topic /turtle1/cmd_vel. Funkcja 
# wykorzystuje utworzony wcześniej obiekt Publishera o nazwie
# twist_publisher.
def move_robot(forward_vel=5,rotation_vel=5):
    '''A function to move turtle from turtlesim simulation
    
    Args:
        forward_vel (float): Linear velocity
        rotation_vel (float): Angular velocity'''
    message=Twist()
    message.angular.z=rotation_vel
    message.linear.x=forward_vel
    
    twist_publisher.publish(message)


In [ ]:
# wywołanie funkcji z przekazaniem argumentów
move_robot(1,-1)


Z punktu widzenia użytkownika konieczne jest dodanie kontrolek umożliwiających wysterowanie
prędkości robota bez znajomości programowania.

# Slider

Wykorzystamy funkcję **move_robot** do utworzenia domyślnego widgetu sterującego prędkościami robota. Każda zmiana wartości kontrolki powoduje wywołanie funkcji **move_robot** i wysłanie odczytanych prędkości z kontrolek. Argumenty funkcji dla, której tworzone będą kontrolki powinny mieć przypisane domyślne wartości. Dla tej funkcji zainicjalizowane domyślnie zostaną dwie kontrolki typu IntSlider. Zakres wartości również jest domyślny. 

Do utworzenia kontrolek wykorzystywana jest funkcja interact z biblioteki ipywidgets (używamy aliasu widgets). Jako argument przekazywana jest funkcja **move_robot**

In [ ]:
widgets.interact(move_robot)


Sterując prędkościami robota, które są w m/s skokowa zmiana o wartości typu Inteeger nie są pożądane. Przydałby się typ zmiennoprzecinkowy. W trakcie tworzenia widgetów można określić oddzielnie dla każdego argumentu funkcji typ kontrolki jaki ma obsługiwać. W tym przypadku zamienimy slidery typu FloatSlider, które umożliwiają ustawianie wartości zmiennoprzecinkowych. 

Pierwszy argument przy wywołaniu funkkcji widgets.interact pozostaje bez zmian, jeśli chodzi o nazwę funkcji. Kolejne argumenty jakie podajemy to nazwa argumentu z funkcji i przypisujemy do niej nową kontrolkę.

**forward_vel=widgets.FloatSlider(min=-10,max=10,value=0)**

Powyższy zapis oznacza, że dla argumentu **forward_vel** z funkcji **move_robot** utworzony zostanie **FloatSlider** z wartością minimalną -10 i maksymalną 10 oraz wartością początkową 0 pomimo tego, że w funkcji **move_robot** ta wartość wynosi 5.

In [ ]:
widgets.interact(move_robot,
                 forward_vel=widgets.FloatSlider(min=-10,max=10,
                                                 value=0),
                 rotation_vel=widgets.FloatSlider(min=-3,max=3,
                                                  value=0))


In [ ]:
# Dokumentację kontrolek można wyświetlić w jupyter notebook
# po wpisaniu typu kotrolki do funkcji help
help(widgets.FloatSlider)


Dla różnych argumentów kontrolki mogą być innego typu np. dla **rotation_vel** utworzona została kontrolka pozwalająca użytkownikowi na wprowadzenie wartości. Argument **step** dla kontrolki **FloatSlider** ustawia krok z jakim mają się zmieniać wartości w kontrolce. 

In [ ]:
widgets.interact(move_robot,
                 forward_vel=widgets.FloatSlider(min=-10,max=10,
                                                 step=2,value=0),
                 rotation_vel=widgets.FloatText(value=0))


Brak podania kontrolki dla argumentu powoduje utworzonie domyślnego slidera.

In [ ]:
widgets.interact(move_robot,
                 rotation_vel=widgets.FloatText(value=0))


# Textbox
Teraz czas na wysłanie dowolnej informacji na topicu **/informacja**. Skorzystanie z kontrolki do wprowadzania tekstu spowoduje, że każde wprowadzenie nowego znaku wyśle wiadomość. W terminalu można podejrzeć informację na tym topicu.

*rostopic echo /informacja*

Oto przykład:

In [ ]:
from std_msgs.msg import String
def send_msg(wiadomosc):
    pub_info = pub_speed=rospy.Publisher("/informacja",String,
                                         queue_size=10)
    msg_info = String()
    msg_info.data = wiadomosc
    pub_info.publish(msg_info)


Tym razem przekazywana jest funkcja **send_msg** oraz uzupełniany jej argument **wiadomosc**.

In [ ]:
widgets.interact(send_msg,
                 wiadomosc=widgets.Text(value='Hello World!'))


Utworzenie kontrolki bez wartości domyślnej dla tekstu. Wartość jest pustym tekstem.

In [ ]:
widgets.interact(send_msg,
                 wiadomosc=widgets.Text())


Odczyt wartości z kontrolki typu **FloatText**. Utworozny został obiekt **liczba** przechowujące dane z kontrolki.

In [ ]:
liczba = widgets.FloatText(
    value=7.5,
    description='Any:',
    disabled=False
)
odczytana_liczba = liczba.value  # odczyt wartości z kontrolki 
# i zapis do zmiennej
print(odczytana_liczba)  # wyświetlenie odczytanej wartości
print("typ wartości: ", type(odczytana_liczba))  # wyświetlenie
# typu odczytanej liczby


In [ ]:
liczba


In [ ]:
liczba.value


# Button

W poprzednim przykładzie każdorazowa zmiana tekstu powoduje jego wysłanie. Z punktu widzenia użytkownika konieczne jest wprowadzenie całego tekstu, a następnie wysłanie go do użytkownika. 

In [ ]:
widgets.ToggleButton(
    value=False,  # domyslna wartość dla przycisku
    description='Przykładowy przycisk',  # opis przyciku
    disabled=False,  # początkowy stan przycisku - wyłączony
    button_style='', # 'success', 'info', 'warning', 'danger'
    # or ''
    tooltip='Description',  # Pojawiający się szczegółowy opis 
    # po najechaniu na przycisk
    icon='check' # (FontAwesome names without the `fa-` prefix)
)


Teraz czas na modyfikację kodu do wysyłania informacji po wciśnięciu przycisku wyślij. W tym celu tworzona jest zmienna *input_text*. Do wyświetlenia kontrolki służy funkcja *display()*, a jako argument przekazywana jest zmienna przechowująca kontrolkę, którą chcemy wyświetlić. W typm przypadku **input_text**.


Teraz kontrolka z tekstem nie jest w żaden sposób zależna z funkcją od wysyłania tekstu.

In [ ]:
input_text = widgets.Text(value='Hello World!', disabled=False, 
                          description="informacja:")
display(input_text)


Odczyt wartości z kontrolki ze zmiennej **input_text** i zapisanie go do zmiennej **odczytany_tekst**, a następnie wyświetlana jest wartość i typ przechowywanej wartości.

In [ ]:
odczytany_tekst = input_text.value
print(odczytany_tekst)
print("typ wartości: ", type(odczytany_tekst))


Funkcja **send_msg** jako argument przyjmuje informację o stanie przycisku **wyslij** opisanego poniżej (patrz description, zmienna *przycisk_wyslij*), z którego pochodzi żądanie wywołania tej funkji po każdorazowym kliknięciu przycisku.

Funkcja **send_msg** wykorzystuje wartość tekstu odczytaną ze zmiennej od kontrolki tekstowej **input_text**.

In [ ]:
from std_msgs.msg import String
def send_msg(button_data):
    pub_info = pub_speed=rospy.Publisher("/informacja",String,
                                         queue_size=10)
    msg_info = String()
    msg_info.data = input_text.value
    pub_info.publish(msg_info)


Utworzenie przycisku do wysyłania wiadomości.

In [ ]:
przycisk_wyslij = widgets.Button(
    value=False,
    description='wyslij',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' 
    # or ''
    tooltip='Description',
    icon='check' # (FontAwesome names without the `fa-` prefix)
)
display(przycisk_wyslij)


W tej chwili po wykonaniu podlądu na topic'u informacja można zobaczyć, że wiadomość nie jest publikowana.

Przycisk wyślij został utworzony, ale brakuje jeszcze obsługi zdarzenia na kliknięcie. Dla utworzonego obiektu przycisku **przycisk_wyslij** wykorzystywana jest metoda **on_click**, która reaguje na kliknięcie przycisku i powoduje wykonanie funkcji **send_msg** podanej w argumencie. Wywołana funkja w argumencie posiada informację związane z przyciskiem od którego pochodzi zdarzenie.

In [ ]:
przycisk_wyslij.on_click(send_msg)


Dopiero po obsłudze zdarzenia na kliknięcie możliwe jest wysyłanie tekstu.

# Checkbox
Przycisk zawierający tylko dwa stany True lub False. Przykład użycia

In [ ]:
checkbox_object = widgets.Checkbox(
    value=False,
    description='Check me',
    disabled=False,
    indent=False
)
display(checkbox_object)


In [ ]:
checkbox_object.value


## Dropdown
Lista rozwijana z opcjami do wyboru

In [ ]:
dropdown = widgets.Dropdown(
    options=['1', '2', '3'],
    value='2',
    description='Number:',
    disabled=False,
)
display(dropdown)


In [ ]:
# Odczyt wartości
dropdown.value


In [ ]:
# Typ wartości
type(dropdown.value)


## RadioButtons
Lista przycisków z opcją do wyboru

In [ ]:
radio_button = widgets.RadioButtons(
    options=['A', 'B', 'C'],
    value='B', # Domyślna wartość
#    layout={'width': 'max-content'}, # If the items' names 
#     are long
    description='Wersja testu:',
    disabled=False
)
display(radio_button)


In [ ]:
# odczyt wartości z kontroli
radio_button.value


In [ ]:
type(radio_button.value)


## Wybór wielu opcji
Lista wyboru wielu opcji

In [ ]:
options = widgets.SelectMultiple(
    options=['skanowanie', 'jazda autonomiczna', 
             'rozpoznawanie piłek tenisowych'],
    value=['skanowanie'],
    #rows=10,
    description='Funkcjonalności robota',
    disabled=False
)
display(options)


In [ ]:
# odczyt wartości
options.value


In [ ]:
# Do pola z opisem opcji przypisanie wartości
options2 = widgets.SelectMultiple(
    options=[('skanowanie',7), ('jazda autonomiczna', 2), 
             ('rozpoznawanie piłek tenisowych', 3)],
    value=[2,3], # zamiast nazw pola podane sa wartości opcji, 
#     które zostały wybrane
    #rows=10,
    description= "Funkkcjonalności robota",
    disabled=False
)
display(options2)


In [ ]:
options2.value


## Pobieranie koloru
Kontrolka do wyboru koloru

In [ ]:
color_picker = widgets.ColorPicker(
    concise=False,
    description='Pick a color',
    value='blue',
    disabled=False
)
display(color_picker)


In [ ]:
# wartość koloru zapisana jako typ tekstowy str w wartości 
# koloru podanego hexadecymalnie
color_picker.value


# Tabs

Do grupowania zakladek i wyświetlania wszystkich w jednej karcie służy **VBox** z omawianej biblioteki. Przyjmuje listę kontrolek do wyświetlenia.

In [ ]:
tab_contents = ['P0', 'P1', 'P2']
# Ustawienie kontrolek w kolejnych zakładkach. Dla zakładek
# tekstowych nadawana jest nazwa taka jak karty. Zmienna children 
# zawiera listę z 3 obiektami VBox, w którym każdy z nich zawiera 
# 3x IntSlider oraz Text
children = [widgets.VBox([widgets.Text(description=name), 
                          widgets.IntSlider(), widgets.IntSlider(),
                          widgets.IntSlider()]) 
            for name in tab_contents]
tab = widgets.Tab(children = children)

# Ustawienie nazw kolejnych zakładek
for i in range(len(tab_contents)):
    tab.set_title(i, tab_contents[i])
# wyświetlenie zakładek
tab


In [ ]:
# Każdą z kontrolek można zdefiniować oddzielnie, a następnie
# można je dodać do odpowiednich zakładek
kontrolka1 = widgets.IntSlider(description="speed")
kontrolka1_2 = widgets.Checkbox(description="slider_source")
kontrolka2 = widgets.IntSlider(description="cos tam")
kontrolka3 = widgets.Text(description="info")
kontrolka3_2 = widgets.IntSlider(description="forward")
kontrolka3_3 = widgets.IntSlider(description="rotational")

# lista z zakładkami
children = [
    widgets.VBox([kontrolka1, kontrolka1_2]), # pierwsza zakładka,
#     zawiera kontrolka1 i kontrolka1_2
    widgets.VBox([kontrolka2]), # druga zakładka zawiera tylko 
#     jedną kontrolkę kontrolka2
    widgets.VBox([kontrolka3, kontrolka3_2, kontrolka3_3])
]

tab = widgets.Tab(children = children)
# ustawianie nazwy zakładek; Kolejne argumenty: id zakładki, 
# nazwa zakładki
tab.set_title(0, "P1")
tab.set_title(1, "P2")
tab.set_title(2, "P3")
tab


### Dodatkowe widgety
https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20List.html

# Zmiana stylu i układu kontrolek
Do opisu wyglądu i stylu kontrolek służy obiekt ipywidgets.Layout.

In [ ]:
## Sprawdzanie dostępnych styli dla kontrolki
b1=widgets.Button(description='To jest przycisk', 
                  layout=widgets.Layout(width='50%', 
                                        height='60px'))
b1.style.keys


## Definiowanie rozmiaru
Na etapie inicjalizacji kontrolki jako argument layout należy przekazać obiekt ipywidgets.Layout z podanym rozmiarem kontrolek. W tym przypadku jest to 50% szerokości oraz 60px wysokości.
Mozna też przekazać argumenty dotyczące maksymalnych i minimalnych rozmiarów. 

Dostępne parametry: 

- height, 

- width, 

- max_height, 

- max_width, 

- min_height, 

- max_height

In [ ]:
przycisk = widgets.Button(description='To jest przycisk',
           layout=widgets.Layout(width='50%', height='60px'))
display(przycisk)
# przycisk zajmuje 50% szerokości obszaru


## Ustawienie siatki do pozycjonowania elementów

In [ ]:
button_layout = widgets.Layout(width='auto', height='auto')
button_style = widgets.ButtonStyle(button_color='darkseagreen')
children = [widgets.Button(layout=button_layout, 
                           style=button_style) for i in range(20)]
grid = widgets.GridBox(children=children,# ustawienie kontrolek 
#                        które będą kolejno wpisane do siatki
        layout=widgets.Layout(
            width='50%',
            # ustawienie szerokości kolejnych kolumn 
            grid_template_columns='100px 50px 100px 40px',
            # ustawienie szerokości kolejnych wierszy, rozmiar 
#             dalszych wierszy jest automatyczny
            grid_template_rows='80px auto 80px 60px',
            # odstęp w pikselach pomiędzy wierszami, 
#             a później kolumnami
            grid_gap='5px 10px')
       )
display(grid)


In [ ]:
# wysyłanie różnych wiadomości tekstowych

from std_msgs.msg import String
def send_msg(wiadomosc):
    pub_info = rospy.Publisher("/informacja",String,queue_size=10)
    msg_info = String()
    msg_info.data = wiadomosc
    pub_info.publish(msg_info)


In [ ]:
layout = widgets.Layout(width='auto', height='auto')
text1 = widgets.Text(value='Tekst 1', layout=layout)
text2 = widgets.Text(value='Tekst 2', layout=layout)
text3 = widgets.Text(value='Tekst 3', layout=layout)
text4 = widgets.Text(value='Tekst 4', layout=layout)
text5 = widgets.Text(value='Tekst 5', layout=layout)


widgets.interact(send_msg, wiadomosc=text1)
widgets.interact(send_msg, wiadomosc=text2)
widgets.interact(send_msg, wiadomosc=text3)
widgets.interact(send_msg, wiadomosc=text4)
widgets.interact(send_msg, wiadomosc=text5)


In [ ]:
children_data=[text1, text2, text3, text4, text5]
grid = widgets.GridBox(children=children_data,
        layout=widgets.Layout(
            width='50%',
            grid_template_columns='150px 150px 150px',
            grid_template_rows='80px auto 80px 60px',
            grid_gap='5px 10px')
       )
display(grid)


## Zmiana kolorów przycisku

In [ ]:
b1 = widgets.Button(description='Custom color')
b1.style.button_color = 'lightgreen'
display(b1)


### Dodatkowe przykłady
https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20Styling.html